Part 0 — Set up

In [1]:
import pandas as pd
import numpy as np

Part 1 — Create two related tables

In [2]:
participants = pd.DataFrame({
    "id": [101, 102, 103, 104, 105, 106],
    "age": [24, 31, 27, 19, 42, 35],
    "group": ["A", "B", "A", "B", "A", "B"]
})

In [3]:
scores = pd.DataFrame({
    "id": [101, 102, 103, 105, 106, 107],
    "score": [81, 94, 88, 91, 85, 79]
})

In [5]:
print(participants)
scores

    id  age group
0  101   24     A
1  102   31     B
2  103   27     A
3  104   19     B
4  105   42     A
5  106   35     B


,id,score
0,101,81
1,102,94
2,103,88
3,105,91
4,106,85
5,107,79


Questions
Which IDs occur in both tables?101,102,103,105,106

Which ID occurs only in participants?104

Which ID occurs only in scores?107

What do you expect an inner join to contain?101,102,103,105,106

Part 2 — Inner join

In [6]:
inner = participants.merge(
    scores,
    on="id",
    how="inner"
)

In [8]:
print(inner)

inner.shape

    id  age group  score
0  101   24     A     81
1  102   31     B     94
2  103   27     A     88
3  105   42     A     91
4  106   35     B     85


(5, 4)

Questions
Which IDs were removed?104,107

Why? Because the are not common to both tables

Did the number of rows match your prediction? Yes

Part 3 — Left join

In [9]:
left = participants.merge(
    scores,
    on="id",
    how="left"
)

In [10]:
left

,id,age,group,score
0,101,24,A,81.0
1,102,31,B,94.0
2,103,27,A,88.0
3,104,19,B,NaN
4,105,42,A,91.0
5,106,35,B,85.0


Which table defines the resulting population?A left join keeps all participant IDs, regardless of whether they appear in scores.

What happened to ID 104?It appears in the result, but its score column is:NaN

Why is its score missing? Because 104 does not exist in the scores table, so there is no matching score to merge in.

Part 4 — Outer join

In [11]:
outer = participants.merge(
    scores,
    on="id",
    how="outer"
)

In [12]:
outer

,id,age,group,score
0,101,24.0,A,81.0
1,102,31.0,B,94.0
2,103,27.0,A,88.0
3,104,19.0,B,NaN
4,105,42.0,A,91.0
5,106,35.0,B,85.0
6,107,NaN,NaN,79.0


In [13]:
right = participants.merge(
    scores,
    on="id",
    how="right"
)

In [14]:
right

,id,age,group,score
0,101,24.0,A,81
1,102,31.0,B,94
2,103,27.0,A,88
3,105,42.0,A,91
4,106,35.0,B,85
5,107,NaN,NaN,79


Part 5 — Diagnose joins with indicator=True

In [15]:
diagnostic = participants.merge(
    scores,
    on="id",
    how="outer",
    indicator=True
)

In [16]:
diagnostic

,id,age,group,score,_merge
0,101,24.0,A,81.0,both
1,102,31.0,B,94.0,both
2,103,27.0,A,88.0,both
3,104,19.0,B,NaN,left_only
4,105,42.0,A,91.0,both
5,106,35.0,B,85.0,both
6,107,NaN,NaN,79.0,right_only


In [17]:
diagnostic["_merge"].value_counts()

_merge
both          5
left_only     1
right_only    1
Name: count, dtype: int64

Questions
How many rows matched?5

How many occurred only in the participant table?1

How many occurred only in the score table?1

Challenge

In [18]:
diagnostic[diagnostic["_merge"] != "both"]


,id,age,group,score,_merge
3,104,19.0,B,NaN,left_only
6,107,NaN,NaN,79.0,right_only


Part 6 — Check key uniqueness

In [19]:
participants["id"].is_unique
scores["id"].is_unique

True

In [20]:
participants["id"].duplicated().sum()
scores["id"].duplicated().sum()

np.int64(0)

Question
Why should this be checked before a one-to-one join?
A one‑to‑one join assumes that the key column (here, id) contains unique values in both tables.

Formally, a one‑to‑one merge requires:

Each key appears at most once in participants AND at most once in scores.
If this assumption is violated, the merge is no longer one‑to‑one, even if you intended it to be.

Part 7 — Validate the merge

In [21]:
validated = participants.merge(
    scores,
    on="id",
    how="left",
    validate="one_to_one"
)

In [22]:
duplicate_scores = pd.DataFrame({
    "id": [101, 101, 102],
    "score": [81, 83, 94]
})

In [23]:
participants.merge(
    duplicate_scores,
    on="id",
    validate="one_to_one"
)

MergeError: Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
  id
101 ...

Question
Why is the error valuable? Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
  id
101 ...

Part 8 — Many-to-many multiplication

In [24]:
visits = pd.DataFrame({
    "id": [1, 1, 2],
    "visit": ["pre", "post", "pre"]
})

treatments = pd.DataFrame({
    "id": [1, 1, 2],
    "treatment": ["A", "B", "A"]
})

In [25]:
visits.merge(
    treatments,
    on="id"
)

,id,visit,treatment
0,1,pre,A
1,1,pre,B
2,1,post,A
3,1,post,B
4,2,pre,A


Questions
Why does ID 1 generate four rows?Why does ID 1 generate four rows?
Because both tables contain duplicate rows for ID 1.

visits has 2 rows with ID 1 treatments has 2 rows with ID 1

Was information duplicated?Yes

Under what circumstances could this be correct? merge many‑to‑many 

Part 9 — Concatenate rows

In [26]:
fall = pd.DataFrame({
    "id": [1, 2, 3],
    "semester": ["Fall"] * 3,
    "score": [80, 85, 90]
})

spring = pd.DataFrame({
    "id": [4, 5, 6],
    "semester": ["Spring"] * 3,
    "score": [78, 92, 88]
})

In [27]:
all_scores = pd.concat(
    [fall, spring],
    axis=0,
    ignore_index=True
)

In [28]:
all_scores

,id,semester,score
0,1,Fall,80
1,2,Fall,85
2,3,Fall,90
3,4,Spring,78
4,5,Spring,92
5,6,Spring,88


Why is concat() more natural than merge() here?

Because the two tables represent different observations, not matching observations.

Part 10 — Concatenating mismatched columns

In [29]:
fall = pd.DataFrame({
    "id": [1, 2],
    "score": [80, 85]
})

spring = pd.DataFrame({
    "id": [3, 4],
    "score": [90, 92],
    "campus": ["Flagstaff", "Flagstaff"]
})

In [30]:
pd.concat(
    [fall, spring],
    ignore_index=True
)

,id,score,campus
0,1,80,NaN
1,2,85,NaN
2,3,90,Flagstaff
3,4,92,Flagstaff


Answer
Pandas adds the missing column (campus) to the fall rows and fills its values with NaN.

This is the correct behavior: concatenation stacks rows, and pandas ensures the resulting DataFrame has a consistent set of columns.

Part 11 — Wide data

In [31]:
wide = pd.DataFrame({
    "id": [101, 102, 103, 104],
    "group": ["A", "B", "A", "B"],
    "score_pre": [72, 85, 79, 88],
    "score_post": [81, 91, 84, 90]
})

In [32]:
wide

,id,group,score_pre,score_post
0,101,A,72,81
1,102,B,85,91
2,103,A,79,84
3,104,B,88,90


Question
How many rows represent each participant?

Exactly one row per participant.

How are repeated measurements represented?

Each repeated measurement becomes a new column rather than a new row.

Part 12 — Wide to long with melt()

In [33]:
long = wide.melt(
    id_vars=["id", "group"],
    value_vars=["score_pre", "score_post"],
    var_name="time",
    value_name="score"
)

In [34]:
long

,id,group,time,score
0,101,A,score_pre,72
1,102,B,score_pre,85
2,103,A,score_pre,79
3,104,B,score_pre,88
4,101,A,score_post,81
5,102,B,score_post,91
6,103,A,score_post,84
7,104,B,score_post,90


Questions

How many rows now represent each participant? Each participant now has two rows

What happened to the original score columns?

The columns:

score_pre

score_post

were collapsed into:

a single column called time (the name of the original column)

a single column called score (the value from the original column)


What does id_vars mean?

id_vars lists the columns that should stay the same and not be melted.

What does value_vars mean?

value_vars lists the columns that should be melted

Part 13 — Clean the melted variable

In [35]:
long["time"] = (
    long["time"]
    .str.replace("score_", "", regex=False)
)

In [36]:
long

,id,group,time,score
0,101,A,pre,72
1,102,B,pre,85
2,103,A,pre,79
3,104,B,pre,88
4,101,A,post,81
5,102,B,post,91
6,103,A,post,84
7,104,B,post,90


In [37]:
long["time"] = (
    long["time"]
    .replace({
        "pre": "baseline",
        "post": "followup"
    })
)


In [38]:
long


,id,group,time,score
0,101,A,baseline,72
1,102,B,baseline,85
2,103,A,baseline,79
3,104,B,baseline,88
4,101,A,followup,81
5,102,B,followup,91
6,103,A,followup,84
7,104,B,followup,90


Part 14 — Long to wide with pivot()

In [39]:
wide_again = long.pivot(
    index=["id", "group"],
    columns="time",
    values="score"
)

In [40]:
wide_again

,time,baseline,followup
id,group,,
101,A,72,81
102,B,85,91
103,A,79,84
104,B,88,90


In [41]:
wide_again = wide_again.reset_index()

In [42]:
wide_again

time,id,group,baseline,followup
0,101,A,72,81
1,102,B,85,91
2,103,A,79,84
3,104,B,88,90


Question
Did you recover the same information as the original wide dataset? Yes

Part 15 — Duplicate combinations and pivot()

In [3]:
repeated = pd.DataFrame({
    "id": [1, 1, 1, 2, 2],
    "time": ["pre", "pre", "post", "pre", "post"],
    "score": [70, 74, 82, 80, 88]
})

NameError: name 'pd' is not defined

In [ ]:
repeated.pivot(
    index="id",
    columns="time",
    values="score"
)

NameError: name 'repeated' is not defined